# 01 — RNN Limitations & Why Attention Matters
### Starter Notebook

**Learning objectives**
- Understand how RNNs process sequences and where they break down
- Experience the vanishing gradient problem first-hand
- Observe the sequential bottleneck that blocks parallelism
- Motivate the move to attention-based models

**Exercises** are marked with `# TODO`. Reference solutions are in `01_rnn_limitations_solution.ipynb`.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Part A — A Minimal RNN

An RNN updates a hidden state $h_t$ at each time step:

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$$

This simple recurrence is powerful — the hidden state is a compressed summary of everything seen so far. But there's a catch.

In [ ]:
class SimpleRNN(nn.Module):
    """A single-layer RNN for next-token prediction."""

    def __init__(self, vocab_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn_cell = nn.RNNCell(hidden_size, hidden_size)
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: token ids of shape (batch, seq_len)
        Returns:
            logits: (batch, seq_len, vocab_size)
        """
        batch, seq_len = x.shape
        embeds = self.embedding(x)          # (batch, seq_len, hidden)
        h = torch.zeros(batch, self.hidden_size, device=x.device)

        outputs = []
        for t in range(seq_len):            # ← sequential loop — cannot parallelise!
            h = self.rnn_cell(embeds[:, t], h)
            outputs.append(self.output(h))

        return torch.stack(outputs, dim=1)  # (batch, seq_len, vocab_size)


rnn = SimpleRNN(vocab_size=100, hidden_size=64).to(device)
x_demo = torch.randint(0, 100, (4, 20), device=device)
logits = rnn(x_demo)
print(f'RNN output shape: {logits.shape}   # (batch=4, seq=20, vocab=100)')

## Part B — The Vanishing Gradient Problem

When we backpropagate through time, gradients are multiplied by the recurrent weight matrix at **every step**. If the largest eigenvalue of $W_h$ is < 1, gradients shrink exponentially and early tokens stop learning.

### Exercise B1
Complete the function below to measure how gradient magnitude changes with sequence depth.

In [ ]:
def measure_gradient_flow(model: nn.Module, seq_lengths: list[int], vocab_size: int = 100):
    """
    For each sequence length, run a forward + backward pass on a random
    sequence and record the gradient norm at the embedding layer.

    Returns:
        List of gradient norms, one per seq_length.
    """
    grad_norms = []

    for seq_len in seq_lengths:
        model.zero_grad()
        x = torch.randint(0, vocab_size, (1, seq_len), device=device)
        targets = torch.randint(0, vocab_size, (1, seq_len), device=device)

        # TODO: forward pass, compute cross-entropy loss, backward pass
        # logits = ...
        # loss = ...
        # loss.backward()

        # TODO: record the gradient norm of model.embedding.weight
        # grad_norm = ...
        grad_norms.append(None)  # replace with grad_norm

    return grad_norms


seq_lengths = [5, 10, 20, 50, 100, 200]
rnn_grads = measure_gradient_flow(rnn, seq_lengths)

plt.figure(figsize=(8, 4))
plt.plot(seq_lengths, rnn_grads, 'o-', color='tomato', label='RNN')
plt.xlabel('Sequence length')
plt.ylabel('Embedding gradient norm')
plt.title('Gradient norm vs sequence length')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Exercise B2 — LSTM vs RNN

LSTMs add a *cell state* and gating to mitigate vanishing gradients. Complete the LSTM model below and add its gradient curve to the plot.

In [ ]:
class SimpleLSTM(nn.Module):
    """Single-layer LSTM for next-token prediction."""

    def __init__(self, vocab_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        # TODO: add an LSTMCell
        self.output = nn.Linear(hidden_size, vocab_size)

    def forward(self, x: torch.Tensor):
        batch, seq_len = x.shape
        embeds = self.embedding(x)
        h = torch.zeros(batch, self.hidden_size, device=x.device)
        c = torch.zeros(batch, self.hidden_size, device=x.device)

        outputs = []
        for t in range(seq_len):
            # TODO: run the LSTM cell and append output projection
            pass

        return torch.stack(outputs, dim=1)


lstm = SimpleLSTM(vocab_size=100, hidden_size=64).to(device)
# lstm_grads = measure_gradient_flow(lstm, seq_lengths)   # uncomment after implementation

# plt.figure(figsize=(8, 4))
# plt.plot(seq_lengths, rnn_grads,  'o-', color='tomato', label='RNN')
# plt.plot(seq_lengths, lstm_grads, 's-', color='steelblue', label='LSTM')
# plt.xlabel('Sequence length'); plt.ylabel('Gradient norm'); plt.yscale('log')
# plt.title('Vanishing gradients: RNN vs LSTM'); plt.legend(); plt.grid(True, alpha=0.3)
# plt.tight_layout(); plt.show()

## Part C — The Sequential Bottleneck

RNNs compute $h_t$ from $h_{t-1}$: each step depends on the previous one. Modern hardware (GPUs) thrives on **parallelism** — the sequential dependency kills utilisation.

### Exercise C1
Measure wall-clock time for the RNN vs a fully parallel operation (matrix multiply) as sequence length grows.

In [ ]:
def time_rnn(seq_len: int, hidden: int = 128, n_runs: int = 20) -> float:
    """Return average forward-pass time in milliseconds."""
    rnn_bench = nn.RNN(hidden, hidden, batch_first=True).to(device)
    x = torch.randn(8, seq_len, hidden, device=device)

    # Warm-up
    with torch.no_grad():
        rnn_bench(x)

    if device.type == 'cuda':
        torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            rnn_bench(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / n_runs * 1000
    return elapsed


def time_matmul(seq_len: int, hidden: int = 128, n_runs: int = 20) -> float:
    """Time a single matrix multiply of comparable FLOP count."""
    x = torch.randn(8, seq_len, hidden, device=device)
    W = torch.randn(hidden, hidden, device=device)

    if device.type == 'cuda':
        torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            x @ W
    if device.type == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / n_runs * 1000


lengths = [32, 64, 128, 256, 512]
rnn_times   = [time_rnn(l)    for l in lengths]
matmul_times = [time_matmul(l) for l in lengths]

plt.figure(figsize=(8, 4))
plt.plot(lengths, rnn_times,    'o-', color='tomato',   label='RNN (sequential)')
plt.plot(lengths, matmul_times, 's-', color='seagreen',  label='MatMul (parallel)')
plt.xlabel('Sequence length')
plt.ylabel('Time per forward pass (ms)')
plt.title('Sequential vs Parallel Computation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part D — The Long-Range Dependency Test

A well-known failure mode of RNNs: copying a token that appears far earlier in the sequence.

### Exercise D1
Build a synthetic task: the model must output the *first* token at the *last* position, regardless of what happens in between. Train an RNN and a simple attention model and compare accuracy.

In [ ]:
def make_copy_task_batch(batch_size: int, seq_len: int, vocab_size: int = 10):
    """
    Generate a 'copy first token' task.
    Input:  [target, noise, noise, ..., noise, <sentinel>]
    Output: only the last position matters — should predict `target`.
    """
    x = torch.randint(1, vocab_size, (batch_size, seq_len))
    target = x[:, 0].clone()   # first token is the answer
    return x, target


# Quick sanity check
xb, yb = make_copy_task_batch(4, 20)
print('Input (first 5 tokens):', xb[:2, :5])
print('Target (last position):', yb[:2])

# TODO (optional challenge):
# 1. Train the SimpleRNN above on this task for seq_len = [10, 30, 50]
# 2. Record final accuracy at each length
# 3. Compare with the MultiHeadAttention from src.models.attention
print('\n[Exercise D1]: implement the training loop and accuracy comparison above.')

## Summary

| Problem | RNN | LSTM | Transformer |
|---------|-----|------|-------------|
| Vanishing gradients | ✗ severe | ~ mitigated | ✓ direct path |
| Sequential computation | ✗ | ✗ | ✓ fully parallel |
| Long-range dependencies | ✗ | ~ | ✓ $O(1)$ path length |
| Scales with compute | limited | limited | ✓ scales well |

**Key insight:** The attention mechanism gives every token a *direct*, differentiable path to every other token — no matter how far apart they are. This is what enables Transformers to model long-range context effectively.

**Next notebook:** `../part2_fundamentals/02_attention_mechanism_starter.ipynb` — we build attention from scratch.